In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import pickle
import zipfile
from pathlib import Path

import cv2
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel
from huggingface_hub import HfApi, login


class Config:
    # Grounding DINO pkl source (zip or folder)
    GDINO_PKL_PATH = "/kaggle/input/grounding-dino-boxes"  # folder with {video_id}.pkl or zip files

    # Raw video root (same videos used in extractvit.ipynb)
    VIDEO_ROOT = "/kaggle/input/raw-casual/root/Desktop/Data/test"
    VIDEO_MISSING = "/kaggle/input/d/danielq07/missing-raw-causal/recovered_clips"

    # DINOv3 backbone (same as extractvit.ipynb for frame features)
    VIT_NAME = "facebook/dinov3-vitl16-pretrain-lvd1689m"
    HIDDEN_DIM = 1024

    # Output
    OUTPUT_DIR = Path("/kaggle/working/grounded_dino_with_roi")

    # Batch size for DINOv3 crops (per frame)
    CROP_BATCH_SIZE = 32

    # Min crop side in pixels before resize
    MIN_CROP_SIDE = 4

    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # HuggingFace (gated model access + upload)
    HF_TOKEN = ""
    HF_REPO_ID = ""  # e.g., "import os
import pickle
import zipfile
from pathlib import Path

import cv2
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel
from huggingface_hub import HfApi, login


class Config:
    # Grounding DINO pkl source (zip or folder)
    GDINO_PKL_PATH = "/kaggle/input/datasets/danielq07/groundingdino-causal/merged_output_box_only"  # folder with {video_id}.pkl or zip files

    # Raw video root (same videos used in extractvit.ipynb)
    VIDEO_ROOT = "/kaggle/input/raw-casual/root/Desktop/Data/test"
    VIDEO_MISSING = "/kaggle/input/d/danielq07/missing-raw-causal/recovered_clips"

    # DINOv3 backbone (same as extractvit.ipynb for frame features)
    VIT_NAME = "facebook/dinov3-vitl16-pretrain-lvd1689m"
    HIDDEN_DIM = 1024

    # Output
    OUTPUT_DIR = Path("/kaggle/working/grounded_dino_with_roi")

    # Batch size for DINOv3 crops (per frame)
    CROP_BATCH_SIZE = 32

    # Min crop side in pixels before resize
    MIN_CROP_SIDE = 4

    # Device
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # HuggingFace (gated model access + upload)
    HF_TOKEN = ""
    HF_REPO_ID = "DanielQ07/kltn"  # e.g., "DanielQ07/kltn"

    # Multi-node support
    NOTEBOOK_ID = 0
    TOTAL_NODES = 1


cfg = Config()
os.environ["HF_TOKEN"] = cfg.HF_TOKEN
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



2026-03-28 22:13:12.271515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774735992.455273      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774735992.508342      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774735992.961444      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774735992.961492      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774735992.961495      23 computation_placer.cc:177] computation placer alr

In [3]:
!pip -q install timm==0.9.12 decord tqdm huggingface_hub transformers accelerate

# ============================================================================
# Load DINOv3 backbone (gated -- requires HF_TOKEN)
# ============================================================================
print(f"Loading {cfg.VIT_NAME}...")
processor = AutoImageProcessor.from_pretrained(cfg.VIT_NAME, token=cfg.HF_TOKEN)
vit_model = AutoModel.from_pretrained(cfg.VIT_NAME, token=cfg.HF_TOKEN)
vit_model.to(cfg.DEVICE)
vit_model.eval()
for p in vit_model.parameters():
    p.requires_grad = False
print(f"Loaded: hidden_size={vit_model.config.hidden_size}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/60.6 kB ? eta -:--:--


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/2.2 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.4 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/13.6 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━ 6.3/13.6 MB 190.4 MB/s eta 0:00:01


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 13.6/13.6 MB 221.7 MB/s eta 0:00:01

...

Loading facebook/dinov3-vitl16-pretrain-lvd1689m...


preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loaded: hidden_size=1024


In [4]:
# ============================================================================
# Helper functions
# ============================================================================

def find_video_path(video_id, video_root):
    video_root = Path(video_root)
    for ext in [".mp4", ".avi", ".mkv", ".mov", ".webm"]:
        path = video_root / f"{video_id}{ext}"
        if path.exists():
            return path
        for subpath in video_root.rglob(f"{video_id}{ext}"):
            return subpath
    return None


def read_frame_at_index(cap, frame_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame_bgr = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


@torch.no_grad()
def extract_roi_features_for_frame(frame_rgb, boxes_xyxy_orig, batch_size=32):
    """
    Crop each box from the frame, pass through DINOv3, extract pooled CLS.

    Args:
        frame_rgb: (H, W, 3) uint8 numpy array
        boxes_xyxy_orig: (N, 4) float32 boxes in original pixel coords
        batch_size: max crops per DINOv3 forward pass

    Returns:
        roi_features: (N, 1024) float32 numpy array
    """
    if len(boxes_xyxy_orig) == 0:
        return np.zeros((0, cfg.HIDDEN_DIM), dtype=np.float32)

    h, w = frame_rgb.shape[:2]
    crops = []

    for box in boxes_xyxy_orig:
        x1, y1, x2, y2 = box.astype(int)
        x1 = max(0, min(x1, w - 1))
        y1 = max(0, min(y1, h - 1))
        x2 = max(x1 + 1, min(x2, w))
        y2 = max(y1 + 1, min(y2, h))

        crop = frame_rgb[y1:y2, x1:x2]
        if crop.shape[0] < cfg.MIN_CROP_SIDE or crop.shape[1] < cfg.MIN_CROP_SIDE:
            crop = np.zeros((224, 224, 3), dtype=np.uint8)
        crops.append(crop)

    all_features = []
    for start in range(0, len(crops), batch_size):
        batch_crops = crops[start : start + batch_size]
        inputs = processor(images=batch_crops, return_tensors="pt")
        inputs = {k: v.to(cfg.DEVICE) for k, v in inputs.items()}
        outputs = vit_model(**inputs)
        # DINOv3: prefer pooler_output (same as extractvit.ipynb)
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            feats = outputs.pooler_output  # (batch, 1024)
        else:
            feats = outputs.last_hidden_state[:, 0]
        all_features.append(feats.cpu().numpy())

    return np.concatenate(all_features, axis=0).astype(np.float32)


def load_gdino_pkls(gdino_path):
    """
    Load all Grounding DINO pkls from a directory or zip files.
    Returns dict: {video_id: package_dict}
    """
    gdino_path = Path(gdino_path)
    packages = {}

    # Check for zip files first
    zip_files = list(gdino_path.glob("*.zip"))
    if zip_files:
        for zf_path in zip_files:
            with zipfile.ZipFile(zf_path, "r") as zf:
                for name in zf.namelist():
                    if name.endswith(".pkl"):
                        vid = Path(name).stem
                        data = pickle.loads(zf.read(name))
                        packages[vid] = data
        print(f"Loaded {len(packages)} pkls from {len(zip_files)} zip files")
    else:
        # Flat pkl files or subdirectories
        for pkl_file in gdino_path.rglob("*.pkl"):
            vid = pkl_file.stem
            with open(pkl_file, "rb") as f:
                packages[vid] = pickle.load(f)
        print(f"Loaded {len(packages)} pkls from directory")

    return packages


print("Helper functions defined")

Helper functions defined


In [5]:
# ============================================================================
# Load all Grounding DINO packages
# ============================================================================
all_packages = load_gdino_pkls(cfg.GDINO_PKL_PATH)
all_video_ids = sorted(all_packages.keys())

# Multi-node split
if cfg.TOTAL_NODES > 1:
    chunk_size = len(all_video_ids) // cfg.TOTAL_NODES
    start = cfg.NOTEBOOK_ID * chunk_size
    end = start + chunk_size if cfg.NOTEBOOK_ID < cfg.TOTAL_NODES - 1 else len(all_video_ids)
    my_video_ids = all_video_ids[start:end]
else:
    my_video_ids = all_video_ids

print(f"Total videos: {len(all_video_ids)}")
print(f"This node ({cfg.NOTEBOOK_ID}): {len(my_video_ids)} videos")

Loaded 26900 pkls from directory
Total videos: 26900
This node (0): 26900 videos


In [6]:
# ============================================================================
# Main extraction loop
# ============================================================================
zip_name = f"gdino_roi_node_{cfg.NOTEBOOK_ID}.zip"
zip_path = cfg.OUTPUT_DIR / zip_name

# Build video path maps (same logic as extractvit.ipynb)
root_map = {p.stem: p for p in Path(cfg.VIDEO_ROOT).rglob("*.mp4")}
miss_map = {p.stem: p for p in Path(cfg.VIDEO_MISSING).rglob("*.mp4")} if os.path.exists(cfg.VIDEO_MISSING) else {}
print(f"Video map: {len(root_map)} root + {len(miss_map)} missing/recovered")

print(f"Creating ZIP: {zip_name}")
zip_file = zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True)

count = 0
fail_count = 0
skipped_no_video = 0

for video_id in tqdm(my_video_ids, desc=f"Node {cfg.NOTEBOOK_ID}"):
    package = all_packages[video_id]
    frames_data = package.get("frames", [])

    if not frames_data:
        fail_count += 1
        continue

    # Try primary root, then missing/recovered (same as extractvit.ipynb)
    video_path = root_map.get(video_id) or miss_map.get(video_id)
    if video_path is None:
        skipped_no_video += 1
        continue

    try:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            fail_count += 1
            continue

        for frame_dict in frames_data:
            frame_idx = frame_dict["frame_idx"]
            boxes_orig = frame_dict.get("boxes_xyxy_orig", np.zeros((0, 4), dtype=np.float32))

            frame_rgb = read_frame_at_index(cap, frame_idx)
            if frame_rgb is None:
                frame_dict["roi_features"] = np.zeros(
                    (len(boxes_orig), cfg.HIDDEN_DIM), dtype=np.float32
                )
                continue

            roi_feats = extract_roi_features_for_frame(
                frame_rgb, boxes_orig, batch_size=cfg.CROP_BATCH_SIZE
            )
            frame_dict["roi_features"] = roi_feats

        cap.release()

        zip_file.writestr(
            f"{video_id}.pkl",
            pickle.dumps(package, protocol=pickle.HIGHEST_PROTOCOL),
        )
        count += 1

    except Exception as e:
        print(f"Error {video_id}: {e}")
        fail_count += 1
        continue

print(f"\nFinalizing ZIP ({count} files, {fail_count} failed, {skipped_no_video} no video)...")
zip_file.close()

file_size_gb = zip_path.stat().st_size / 1e9
print(f"Final size: {file_size_gb:.2f} GB")

Video map: 24173 root + 2743 missing/recovered
Creating ZIP: gdino_roi_node_0.zip


Node 0:   0%|          | 0/26900 [00:00<?, ?it/s]

In [ ]:
# ============================================================================
# Upload to HuggingFace
# ============================================================================
if cfg.HF_TOKEN and cfg.HF_REPO_ID:
    login(token=cfg.HF_TOKEN)
    api = HfApi()
    api.create_repo(cfg.HF_REPO_ID, repo_type="dataset", exist_ok=True, token=cfg.HF_TOKEN)

    print(f"Uploading {zip_name}...")
    api.upload_file(
        path_or_fileobj=str(zip_path),
        path_in_repo=f"roi_features/{zip_name}",
        repo_id=cfg.HF_REPO_ID,
        repo_type="dataset",
        token=cfg.HF_TOKEN,
    )
    print("Upload complete!")
    zip_path.unlink()
else:
    print("HF credentials not set, skipping upload.")
    print(f"Output saved at: {zip_path}")

In [ ]:
# ============================================================================
# Verification: load one pkl and check roi_features
# ============================================================================
with zipfile.ZipFile(zip_path if zip_path.exists() else cfg.OUTPUT_DIR / zip_name, "r") as zf:
    names = [n for n in zf.namelist() if n.endswith(".pkl")]
    if names:
        sample_pkg = pickle.loads(zf.read(names[0]))
        vid = sample_pkg["video_id"]
        print(f"Video: {vid}")
        print(f"Num frames: {len(sample_pkg['frames'])}")
        for i, fr in enumerate(sample_pkg["frames"][:3]):
            roi = fr.get("roi_features", None)
            boxes = fr.get("boxes_xyxy_orig", np.zeros((0, 4)))
            labels = fr.get("labels_text", [])
            print(f"  Frame {i}: {len(boxes)} boxes, "
                  f"roi_features={'None' if roi is None else roi.shape}, "
                  f"labels={labels[:5]}")
    else:
        print("No pkl files in zip!")